## 1. Table II: Objective comparison

- Scenarios: Vehicle, Cell, Graph
- Policy: Original/Inverted
- RSUs: 382, 422 (10%)

### 1.1. Baseline methods (dwell-ranked (dense), random)

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path("../deploy/BASELINES/results/baselines")

# (representation, rsu budget)
TARGETS = [
    ("vehicle", 382),
    ("cell",    422),
    ("graph",   422),
]

for method in ("dense", "random"):
    for representation, rsus in TARGETS:
        for policy in ("", "inverted_"):

            name = f"{representation}_{policy.rstrip('_')}" if policy else representation
            # -> "vehicle", "vehicle_inverted", "cell", "cell_inverted", ...

            path = BASE / f"{name}_{method}.csv"
            if not path.is_file():
                print(f"[skip] {path} not found")
                continue

            df = pd.read_csv(path, sep=";")

            row = df[df["rsus"] == rsus]
            if row.empty:
                print(f"[skip] {name} {method}: no row for rsus={rsus}")
                continue

            sat_cols = ["c0_s", "c1_s", "c2_s"]
            worst_per_run = row[sat_cols].min(axis=1)

            print(f"Representation: {representation}")
            print(f"Policy: {'original' if not policy else policy.rstrip('_')}")
            print(f"Method: {method}  |  rsus = {rsus}")

            if method == "dense":
                # single deterministic row
                print(f"  obj (min across clusters): {worst_per_run.iloc[0]:.4f}")
            else:
                # one row per seed -> aggregate across seeds
                print(f"  obj mean : {worst_per_run.mean():.4f}")
                print(f"  obj min  : {worst_per_run.min():.4f}")
                print(f"  obj max  : {worst_per_run.max():.4f}")
                print(f"  obj std  : {worst_per_run.std():.4f}")
                print(f"  n seeds  : {len(worst_per_run)}")

            print()

Representation: vehicle
Policy: original
Method: dense  |  rsus = 382
  obj (min across clusters): 0.6204

Representation: vehicle
Policy: inverted
Method: dense  |  rsus = 382
  obj (min across clusters): 0.7407

Representation: cell
Policy: original
Method: dense  |  rsus = 422
  obj (min across clusters): 0.6058

Representation: cell
Policy: inverted
Method: dense  |  rsus = 422
  obj (min across clusters): 0.7116

Representation: graph
Policy: original
Method: dense  |  rsus = 422
  obj (min across clusters): 0.7099

Representation: graph
Policy: inverted
Method: dense  |  rsus = 422
  obj (min across clusters): 0.6551

Representation: vehicle
Policy: original
Method: random  |  rsus = 382
  obj mean : 0.5735
  obj min  : 0.5612
  obj max  : 0.5834
  obj std  : 0.0045
  n seeds  : 30

Representation: vehicle
Policy: inverted
Method: random  |  rsus = 382
  obj mean : 0.3310
  obj min  : 0.2920
  obj max  : 0.3765
  obj std  : 0.0231
  n seeds  : 30

Representation: cell
Policy: ori

### 1.2 SA

In [2]:
import pandas as pd
from pathlib import Path

SA_ROOT = Path("../deploy/testSA")

# (experiment folder, scenario inside the summary CSV, rsu budget)
SA_TARGETS = [
    (SA_ROOT / "veic_res",                "vehicle", 382),
    (SA_ROOT / "veic_res_inverted",       "vehicle", 382),
    (SA_ROOT / "cell_graph_res",          "cell",    422),
    (SA_ROOT / "cell_graph_res_inverted", "cell",    422),
    (SA_ROOT / "cell_graph_res",          "graph",   422),
    (SA_ROOT / "cell_graph_res_inverted", "graph",   422),
]

for folder, scenario, rsus in SA_TARGETS:

    summary_path = folder / f"rsu_{rsus:04d}" / "tuning_summary.csv"
    if not summary_path.is_file():
        print(f"[skip] {summary_path} not found\n")
        continue

    df = pd.read_csv(summary_path, sep=";")

    row = df[df["scenario"].str.lower() == scenario.lower()]
    if row.empty:
        print(f"[skip] {summary_path}: no row for scenario='{scenario}'\n")
        continue

    mean = float(row["best_objective_mean"].iloc[0])
    std  = float(row["best_objective_std"].iloc[0])
    runs = int(row["successful_runs"].iloc[0])

    print(f"Representation: {scenario}")
    print(f"Policy: {'inverted' if 'inverted' in folder.name else 'original'}")
    print(f"Method: SA  |  rsus = {rsus}")
    print(f"  obj mean : {mean:.4f}")
    print(f"  obj std  : {std:.4f}")
    print(f"  mean ± std: {mean:.4f} ± {std:.4f}")
    print(f"  runs     : {runs}")
    print()

Representation: vehicle
Policy: original
Method: SA  |  rsus = 382
  obj mean : -0.7150
  obj std  : 0.0038
  mean ± std: -0.7150 ± 0.0038
  runs     : 30

Representation: vehicle
Policy: inverted
Method: SA  |  rsus = 382
  obj mean : -0.8163
  obj std  : 0.0049
  mean ± std: -0.8163 ± 0.0049
  runs     : 30

Representation: cell
Policy: original
Method: SA  |  rsus = 422
  obj mean : -0.7350
  obj std  : 0.0057
  mean ± std: -0.7350 ± 0.0057
  runs     : 30

Representation: cell
Policy: inverted
Method: SA  |  rsus = 422
  obj mean : -0.7997
  obj std  : 0.0050
  mean ± std: -0.7997 ± 0.0050
  runs     : 30

Representation: graph
Policy: original
Method: SA  |  rsus = 422
  obj mean : -0.7672
  obj std  : 0.0064
  mean ± std: -0.7672 ± 0.0064
  runs     : 30

Representation: graph
Policy: inverted
Method: SA  |  rsus = 422
  obj mean : -0.8039
  obj std  : 0.0061
  mean ± std: -0.8039 ± 0.0061
  runs     : 30



## 2. Table III: Metrics and group utility

- Scenario: Cell
- Policy: Original
- RSUs: 422 (10%)

In [3]:
import import_first
from common import deploy_utils, feature_utils
import pandas as pd

def calculate_cluster_metrics(
    vehicle_metrics: pd.DataFrame,
    clusters: pd.DataFrame,
    N_required: list[float],
    Tn_required: list[float],
) -> pd.DataFrame:
    """
    Calculate aggregate metrics and the fraction of vehicles in each
    cluster that meet the required N and Tn thresholds.

    Parameters
    ----------
    vehicle_metrics : pd.DataFrame
        Must contain:
            vehicle_id, N, Tn, Tc

    clusters : pd.DataFrame
        Must contain:
            vehicle_id, cluster

    N_required : list[float]
        Required N threshold for each cluster.
        N_required[k] corresponds to cluster k.

    Tn_required : list[float]
        Required Tn threshold for each cluster.
        Tn_required[k] corresponds to cluster k.

    Returns
    -------
    pd.DataFrame
        One row per cluster containing:
            number_of_vehicles
            mean_N
            mean_Tn
            mean_Tc
            fraction_N_meeting
            fraction_Tn_meeting
            fraction_N_Tn_meeting
    """

    if len(N_required) != len(Tn_required):
        raise ValueError(
            "N_required and Tn_required must have the same length."
        )

    df = vehicle_metrics.merge(clusters, on="vehicle_id", how="left", validate="one_to_one")

    if df["cluster"].isna().any():
        missing = df.loc[
            df["cluster"].isna(),
            "vehicle_id"
        ].tolist()

        raise ValueError(
            f"Vehicles without cluster assignment: {missing[:10]}"
        )

    # Cluster-specific requirements.
    df["N_required"] = df["cluster"].astype(int).map(
        dict(enumerate(N_required))
    )

    df["Tn_required"] = df["cluster"].astype(int).map(
        dict(enumerate(Tn_required))
    )

    # Vehicle-level threshold satisfaction.
    df["N_meets"] = df["N"] >= df["N_required"]
    df["Tn_meets"] = df["Tn"] <= df["Tn_required"]
    df["N_Tn_meets"] = df["N_meets"] & df["Tn_meets"]

    result = (
        df
        .groupby("cluster")
        .agg(
            number_of_vehicles=("vehicle_id", "size"),
            mean_N=("N", "mean"),
            mean_Tn=("Tn", "mean"),
            mean_Tc=("Tc", "mean"),
            fraction_N_meeting=("N_meets", "mean"),
            fraction_Tn_meeting=("Tn_meets", "mean"),
            fraction_N_Tn_meeting=("N_Tn_meets", "mean")
        )
        .sort_index()
    )

    return result

def calculate_satisfaction(cluster_metrics: pd.DataFrame, N_required: list[float], Tn_required: list[float], alpha: float) -> pd.DataFrame:
    """
    Reproduce the C satisfaction calculation:

        S_N  = min(N / N_required, 1)

        S_Tn = min(Tn_required / Tn, 1)

        S = alpha * S_N + (1-alpha) * S_Tn
    """

    result = cluster_metrics.copy()

    for cluster in result.index:

        k = int(cluster)

        mean_N = result.loc[ cluster, "mean_N" ]
        mean_Tn = result.loc[ cluster, "mean_Tn" ]

        N_req = N_required[k]
        Tn_req = Tn_required[k]

        if N_req <= 0:
            N_satisfaction = 1.0
        else:
            N_satisfaction = min( mean_N / N_req, 1.0)

        if mean_Tn <= 0:
            Tn_satisfaction = 0.0
        else:
            Tn_satisfaction = min( Tn_req / mean_Tn, 1.0)

        satisfaction = ( alpha * N_satisfaction + (1.0 - alpha) * Tn_satisfaction)

        result.loc[ cluster, "N_satisfaction" ] = N_satisfaction
        result.loc[ cluster, "Tn_satisfaction" ] = Tn_satisfaction
        result.loc[ cluster, "satisfaction" ] = satisfaction

    return result

In [4]:
trace = deploy_utils.Trace.read_trace("../../data/6_to_8am.csv")

In [5]:
import pandas as pd
cell_clusters = pd.read_csv("../../src/cells/vc_from_c.csv", header=None, names=["vehicle_id", "cluster"], sep=";")
graph_clusters = pd.read_csv("../../src/cells/vc_from_g.csv", header=None, names=["vehicle_id", "cluster"], sep=";")

In [6]:
dwell_rank_422_dep = deploy_utils.RSUs.read_deployment(
    "../deploy/BASELINES/results/baselines/c_and_g/dense/rsu_0422.csv")

In [7]:
sa_tuning_raw_422 = pd.read_csv("../deploy/testSA/cell_res_keep_dep/rsu_0422/tuning_raw.csv", sep=";")
sa_tuning_raw_422.sort_values(by="best_objective", ascending=True).head(n=1)

,scenario,N,Tn,rsus,seed,best_objective,best_c0_s,best_c1_s,best_c2_s
11,cell,"(2.0, 1.0, 3.0)","(90.0, 120.0, 60.0)",422,12,-0.743617,0.745466,0.787244,0.743617


In [8]:
best_sa_422_dep = deploy_utils.RSUs.read_deployment(
    "../deploy/testSA/cell_res_keep_dep/rsu_0422/cell/c_and_g_trace_vc_from_c_dep_422_2_1_3_90_120_60_10000_1.000_0.990_0.50_12.csv"
)

### 2.1. Dwell results

In [9]:
calculate_satisfaction(
    cluster_metrics=calculate_cluster_metrics(
        vehicle_metrics=deploy_utils.Metrics.get_vehicles_N_Tn_Tc(trace=trace, rsus=dwell_rank_422_dep),
        clusters=cell_clusters,
        N_required=(2, 1, 3),Tn_required=(90, 120, 60)),
    N_required=(2, 1, 3), Tn_required=(90, 120, 60),
    alpha=0.5
)

,number_of_vehicles,mean_N,mean_Tn,mean_Tc,fraction_N_meeting,fraction_Tn_meeting,fraction_N_Tn_meeting,N_satisfaction,Tn_satisfaction,satisfaction
cluster,,,,,,,,,,
0,25757,23.070427,113.417712,10.484528,0.996933,0.496913,0.494468,1.0,0.793527,0.896763
1,32021,9.273008,122.544268,13.067987,0.941382,0.598795,0.579026,1.0,0.979238,0.989619
2,17738,7.200981,283.497012,12.327207,0.656331,0.022720,0.009584,1.0,0.211642,0.605821


### 2.2. SA results

In [10]:
calculate_satisfaction(
    cluster_metrics=calculate_cluster_metrics(
        vehicle_metrics=deploy_utils.Metrics.get_vehicles_N_Tn_Tc(trace=trace, rsus=best_sa_422_dep),
        clusters=cell_clusters,
        N_required=(2, 1, 3),Tn_required=(90, 120, 60)),
    N_required=(2, 1, 3), Tn_required=(90, 120, 60),
    alpha=0.5
)

,number_of_vehicles,mean_N,mean_Tn,mean_Tc,fraction_N_meeting,fraction_Tn_meeting,fraction_N_Tn_meeting,N_satisfaction,Tn_satisfaction,satisfaction
cluster,,,,,,,,,,
0,25757,7.362154,183.324921,13.472842,0.925185,0.123345,0.111193,1.0,0.490932,0.745466
1,32021,3.547547,208.881359,15.077918,0.877643,0.307892,0.281253,1.0,0.574489,0.787244
2,17738,8.026215,123.144097,11.650693,0.883076,0.076841,0.053839,1.0,0.487234,0.743617


## 3. Individual compliance for graph scenario (J_k(D))

* Scenario: graph
* Policy: original
* RSUs: 1056

In [11]:
dwell_rank_1056_dep = deploy_utils.RSUs.read_deployment("../deploy/BASELINES/results/baselines/c_and_g/dense/rsu_1056.csv")

### 3.1. Dwell results

In [12]:
calculate_satisfaction(
    cluster_metrics=calculate_cluster_metrics(
        vehicle_metrics=deploy_utils.Metrics.get_vehicles_N_Tn_Tc(trace=trace, rsus=dwell_rank_1056_dep),
        clusters=graph_clusters,
        N_required=(1, 2, 3),Tn_required=(120, 90, 60)),
    N_required=(1, 2, 3), Tn_required=(120, 90, 60),
    alpha=0.5
)

,number_of_vehicles,mean_N,mean_Tn,mean_Tc,fraction_N_meeting,fraction_Tn_meeting,fraction_N_Tn_meeting,N_satisfaction,Tn_satisfaction,satisfaction
cluster,,,,,,,,,,
0,6234,26.308470,102.871351,10.351299,0.968560,0.718479,0.707732,1.0,1.0,1.0
1,24037,27.017307,76.678454,10.792528,0.976287,0.709614,0.698756,1.0,1.0,1.0
2,45245,18.655255,56.126644,10.755664,0.966162,0.704056,0.689734,1.0,1.0,1.0


### 3.2. SA results

In [13]:
sa_tuning_raw_1056 = pd.read_csv("../deploy/testSA/graph_res_keep_dep/rsu_1056/tuning_raw.csv", sep=";")
sa_tuning_raw_1056.sort_values(by="best_objective", ascending=True).head(n=1)

,scenario,N,Tn,rsus,seed,best_objective,best_c0_s,best_c1_s,best_c2_s
0,graph,"(1.0, 2.0, 3.0)","(120.0, 90.0, 60.0)",1056,1,-1.0,1.0,1.0,1.0


In [14]:
best_sa_1056_dep = deploy_utils.RSUs.read_deployment(
    "../deploy/testSA/graph_res_keep_dep/rsu_1056/graph/c_and_g_trace_vc_from_g_dep_1056_1_2_3_120_90_60_10000_1.000_0.990_0.50_1.csv"
)

In [15]:
calculate_satisfaction(
    cluster_metrics=calculate_cluster_metrics(
        vehicle_metrics=deploy_utils.Metrics.get_vehicles_N_Tn_Tc(trace=trace, rsus=best_sa_1056_dep),
        clusters=graph_clusters,
        N_required=(1, 2, 3),Tn_required=(120, 90, 60)),
    N_required=(1, 2, 3), Tn_required=(120, 90, 60),
    alpha=0.5
)

,number_of_vehicles,mean_N,mean_Tn,mean_Tc,fraction_N_meeting,fraction_Tn_meeting,fraction_N_Tn_meeting,N_satisfaction,Tn_satisfaction,satisfaction
cluster,,,,,,,,,,
0,6234,12.782323,105.542188,10.577478,0.983959,0.741578,0.731954,1.0,1.0,1.0
1,24037,15.411366,89.960478,10.960602,0.976328,0.633315,0.618962,1.0,1.0,1.0
2,45245,13.507570,59.926622,10.948834,0.966648,0.644712,0.620687,1.0,1.0,1.0
